[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MLManuals/Deep-Learning/blob/main/chapter10_building_a_network_with_nn_module/notebook.ipynb)

# Chapter 10: Building a Network with nn.Module

Run the cell below once, first, to install any missing packages (only needed on Google Colab; skip it if you're running locally and already have them).

In [ ]:
%pip install -q torch numpy scikit-learn

The network in Chapter 9 had four loose tensors in it, called `W1`, `b1`, `W2` and `b2`, and you had to pass all four into every function by hand. That was already slightly annoying for nine weights. A network with ten layers would have twenty such variables, and the AlexNet of Chapter 1 would have had hundreds of them, each one needing to be created, initialised, passed around, handed to the optimizer and saved to disk without a single one being forgotten.

In March 2015 a researcher at Google named François Chollet released a library called Keras, built on a simple idea that now seems obvious. A layer should be an object. It should know its own weights, create them at the right size, and expose one way to be called. Building a network should then be a matter of stacking those objects, and in Keras it famously took one line per layer.

PyTorch has the same idea in a class called `nn.Module`, with one extra turn of the screw that makes it more powerful than it first looks. A module can be a single layer, or a group of layers, or an entire network, and modules can contain other modules. The same object that describes one linear layer also describes a hundred layer model, which means the whole thing nests.

This chapter turns your loose tensors into a proper model. Before you read on, try to guess the answers to these four questions.

1. **If a layer created its own weights, what would it need to know in advance?**
2. **What should a network object be able to tell you about itself?**
3. **Why might it help that a layer and a whole network are the same kind of thing?**
4. **How does an optimizer find out which tensors it is allowed to change?**

## Layers the PyTorch Way

Every hidden layer you have written has done the same two things, which are a matrix multiplication and a bias addition. PyTorch calls that a **linear layer**, and `nn.Linear` builds one for you. You tell it how many numbers come in and how many go out, exactly the shape rule from Chapter 3.

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(7)
layer = nn.Linear(2, 3)
print(layer)
print(layer.weight.shape, layer.bias.shape)

Those are the shapes you worked out by hand in Chapter 3, where a layer taking 2 inputs to 3 neurons needed a 3 by 2 weight matrix and 3 biases. The layer made them itself. It also filled them with small random starting values, so you never have to think about where a network begins, and it switched on `requires_grad` for both.

In [ ]:
print(layer.weight.data.round(decimals=3))
print(layer.weight.requires_grad)

Calling the layer runs the same arithmetic you wrote by hand. The next cell feeds it the student from Chapter 3 who studied for 8 hours and slept 3, wrapped in `no_grad` because this is just a look rather than part of training.

In [ ]:
student = torch.tensor([8.0, 3.0])
with torch.no_grad():
    print(layer(student).round(decimals=3))

Those numbers are different from Chapter 3's because the weights are random rather than the ones you chose, but the operation is identical. Figure 10.1 shows what `nn.Linear(2, 3)` quietly took care of.

> 🖼️ **Figure 10.1:** What a single linear layer does for you. Every one of these four jobs was something you did by hand in Part 2.

**A layer object knows its own shape, creates and initialises its own weights, and is ready for autograd, so the only thing you have to decide is how many numbers go in and how many come out.**

## Writing an nn.Module

A layer on its own is not a network. To build one, you write a class that inherits from `nn.Module` and fills in two methods. The first, `__init__`, creates the pieces. The second, `forward`, says how an input flows through them, which is exactly the forward pass of Chapter 3.

In [ ]:
class TinyNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(2, 8)
        self.output = nn.Linear(8, 1)

    def forward(self, x):
        h = torch.relu(self.hidden(x))
        return self.output(h)

That is the whole model. The `super().__init__()` line is required, because it lets `nn.Module` set up the bookkeeping that finds your layers later, and forgetting it is a common first mistake. Notice that `forward` reads like the pencil and paper description from Chapter 3, which is a hidden layer with a ReLU bend and then an output layer.

In [ ]:
torch.manual_seed(7)
net = TinyNet()
print(net)

Printing a model prints its structure, which is genuinely useful when you are checking that a network is what you meant to build. One rule matters here. You call the model, not its `forward` method, writing `net(x)` rather than `net.forward(x)`, because calling the object runs some necessary work around `forward` that you would otherwise skip.

This network has no activation on its output, which is deliberate. Chapter 4's table said a network predicting a plain number should leave its output alone, and Chapter 11 will show that the same is true for classification in PyTorch, for a reason that surprises most beginners.

## Putting Layers Together

When a network is just a straight run of layers in order, writing a class is more ceremony than it deserves. `nn.Sequential` takes the layers as arguments and chains them for you, with each one receiving the previous one's output.

In [ ]:
torch.manual_seed(7)
deep = nn.Sequential(
    nn.Linear(2, 3),
    nn.ReLU(),
    nn.Linear(3, 3),
    nn.ReLU(),
    nn.Linear(3, 1))
print(deep)

That is the two inputs, two hidden layers of three, one output network drawn in Figure 3.2, written in five lines. The activations appear as layers of their own here, because in PyTorch a ReLU can be either a function you call or an object you stack, and inside `Sequential` it has to be an object.

The nesting promised at the start of this chapter is what makes this scale. A `Sequential` is itself a module, so it can sit inside a class alongside other layers, and a block you define once can be reused many times. Figure 10.2 shows the idea, which is how networks with hundreds of layers stay readable.

> 🖼️ **Figure 10.2:** Modules inside modules. A block you write once can be reused, and the whole model is itself a module, which is how very deep networks stay manageable.

## Counting the Parameters

A module keeps a register of every weight and bias inside it, including those in any nested modules, and `parameters()` hands them over. Counting them is the same arithmetic you did by hand in Chapter 3, now done for you.

In [ ]:
print(sum(p.numel() for p in net.parameters()))

Thirty three. That is the same count Chapter 1 reported for the network that learned the plants in the field, and the same number Chapter 3's counting cell produced for the widths `[2, 8, 1]`. Three different routes, one answer. Asking for names as well as values shows exactly where those 33 live.

In [ ]:
for name, p in net.named_parameters():
    print(name, tuple(p.shape), p.numel())

Sixteen weights from the two inputs to the eight hidden neurons, eight biases there, eight weights on to the output and one bias at the end. That is word for word the breakdown Chapter 3 gave. The deeper `Sequential` agrees with Chapter 3 too.

In [ ]:
print(sum(p.numel() for p in deep.parameters()))

Twenty five, exactly as Table 3.1 counted connection by connection. Table 10.1 lays the two together.

**Table 10.1: The parameter counts you worked out by hand, confirmed by the model itself.**

| Network | Counted by hand, Chapter 3 | Reported by the module | Also seen in |
|---|---|---|---|
| 2, 8, 1 | 33 | 33 | Chapter 1's plants |
| 2, 3, 3, 1 | 25 | 25 | Figure 3.2 |

That register exists for a practical reason, which answers the fourth opening question. An **optimizer** is the object that performs the downhill step from Chapter 6, and it needs to know which tensors it is allowed to change. You hand it `parameters()` and it takes care of the rest.

In [ ]:
optimizer = torch.optim.SGD(net.parameters(), lr=0.1)
print(type(optimizer).__name__, optimizer.param_groups[0]["lr"])

SGD stands for stochastic gradient descent, and for now it does exactly what you wrote by hand in Chapter 6, which is to subtract the learning rate times the gradient from every parameter. Chapter 13 explains the word stochastic and introduces better optimizers. Two methods matter, and together they replace the fiddly bookkeeping from Chapter 9.

- **`optimizer.zero_grad()`** clears every gradient, which is the chore that Chapter 9 warned would otherwise let gradients pile up to 2, then 4, then 6.
- **`optimizer.step()`** applies the downhill step to every parameter at once, replacing the `with torch.no_grad()` block you wrote by hand.

**A module registers every parameter inside it, which is what lets you count them, hand them to an optimizer, and later save and load them, without tracking a single tensor yourself.**

## Practice and Review

Reading about deep learning is only half of learning it. Pulling an idea back out of your memory makes it stick much better than reading it again, so this section gives you two kinds of practice, one for your hands and one for your memory.

### Try It Yourself

1. **Forget the super call.** Delete `super().__init__()` from `TinyNet`, build it again and read the error. That error message is worth recognising, because you will see it again.
2. **Widen the network.** Change `TinyNet` to use 16 hidden neurons instead of 8, count its parameters, and check the total against the formula from Chapter 3.
3. **Rebuild XOR as a module.** Write the Chapter 3 XOR network as an `nn.Sequential` with shapes 2, 2, 1, print it, and count its parameters. Compare that count with the nine weights and biases you set by hand.

### Check Your Understanding

Try to answer each question from memory before you look at the answers that follow.

1. **What four jobs does `nn.Linear` do when you create it?**
2. **Which two methods does an `nn.Module` normally define, and what does each one do?**
3. **Why should you call `net(x)` rather than `net.forward(x)`?**
4. **What does it mean that modules nest?**
5. **What does an optimizer need from a model, and why?**

### Answers

1. It creates a weight matrix of the right shape, creates the biases, fills both with small random values, and switches on `requires_grad`.
2. `__init__` creates the layers, and `forward` says how an input flows through them.
3. Because calling the object runs necessary work around `forward` that calling `forward` directly would skip.
4. That a module can contain other modules, so a layer, a reusable block and a whole network are all the same kind of thing.
5. It needs `parameters()`, the register of every weight and bias, so it knows which tensors it is allowed to change.

## What Comes Next

You can now build a model as an object that owns its layers, create it in a few readable lines, print its structure, count its parameters and hand them to an optimizer.

- **Chapter 11, The Training Loop,** joins the model, the loss, the optimizer and your data into the five step loop you will use for the rest of this book.
- **Chapter 12, Datasets, DataLoaders and Batching,** feeds that loop in small batches rather than all the data at once.

Chapter 11 is where all of this finally pays off, because you train a network on real measurements for the first time, taken from 150 flowers picked in Canada in the 1930s.